# Queries, closed forms, audit, and confidence intervals

Interventional probabilities, ATE, PN/PS/PNS, custom linear combinations,
`try_closed_form`, `audit_bound`, and `bound_ci`.


In [ ]:
from partialid import (
    Assumptions, Query, Variables,
    audit_bound, bound, bound_ci, try_closed_form,
)
from partialid.query_ast import Atom, LinComb, Lit, PO

V = Variables.from_binary("X", "Y")
P = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
a = Assumptions.order("X", "Y")


## Built-in query faces


In [ ]:
queries = {
    "interventional": Query.interventional(outcome="Y", do={"X": 1}, value=1),
    "ATE": Query.ate(outcome="Y", treatment="X"),
    "PNS": Query.pns(),
    "PN": Query.pn(),
    "PS": Query.ps(),
}
for name, Q in queries.items():
    r = bound(V, P, Q, a)
    print(f"{name:14s}  {Q.describe()}")
    print(f"{'':14s}  [{r.lower:.4f}, {r.upper:.4f}]  w={r.width:.4f}  method={r.method}\n")


## Custom linear combination via the query AST


In [ ]:
# 2 P(Y_1=1) - P(Y_0=1)
expr = LinComb([
    (2.0, Atom(Lit(PO("Y", {"X": 1}), 1))),
    (-1.0, Atom(Lit(PO("Y", {"X": 0}), 1))),
])
Q_custom = Query.from_expr(expr)
r_custom = bound(V, P, Q_custom, a)
print(Q_custom.describe())
print(r_custom.summary(Q_custom))


## Conditional interventional


In [ ]:
V3 = Variables.from_binary("W", "X", "Y")
P3 = {c: 0.125 for c in V3.cells()}
Qc = Query.interventional(outcome="Y", do={"X": 1}, value=1, condition={"W": 0})
print(bound(V3, P3, Qc, Assumptions.order("W", "X", "Y")).summary(Qc))


## Closed-form shortcuts


In [ ]:
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
cf = try_closed_form(V, P, Q, Assumptions.manski("X", "Y"))
print(cf)
if cf is not None:
    print(f"closed interval=[{cf.lower:.4f}, {cf.upper:.4f}]  id={cf.formula_id}")


## Audit a bound


In [ ]:
assumptions = a & Assumptions.mtr("Y", "X") & Assumptions.tan_msm("X", 2.0)
r = bound(V, P, Q, assumptions)
report = audit_bound(r, query=Q, assumptions=assumptions)
print(report)
print("faces:", report.faces)
print("budgets:", report.budgets)


## Imbens–Manski-style CI for the identified set


In [ ]:
ci = bound_ci(V, P, Q, a, n_boot=50, n=500, seed=0)
print(ci)
print(
    f"point=[{ci.point_lower:.4f}, {ci.point_upper:.4f}]  "
    f"{100 * ci.level:.0f}% CI=[{ci.lower:.4f}, {ci.upper:.4f}]"
)
